# Granite-CLM-Preview-3B — Lossless CLM Lift & Release

This notebook lifts `ibm-granite/granite-3.1-3b-a800m-base` into the canonical CLM MoE substrate, verifies byte/forward/router/generation parity, records release metrics and provenance, publishes the verified bundle under `archelabs-org/native-clm-v0/granite-clm-preview-3b/`, and pushes lightweight evidence back to GitHub.

**Release boundary:** no training or weight mutation occurs. Expert-slice addresses are deterministic mutation coordinates only; this release does not claim that pretrained Experts are Cells, nor that safe evolution/composability/replay-free learning is solved.

**Validated loader environment:** this release pins `transformers==4.47.0`, matching the version recorded by the Granite 3.1 3B-A800M checkpoint itself. This keeps the release on the model's native Transformers generation while avoiding the newer 5.x weight-materialization path that stalled in the first hosted run.

**GPU policy:** the release intentionally uses one GPU (`cuda:0`) even when Kaggle exposes two T4s. The 3B FP16 model fits on one T4-class GPU; the release workload is dominated by checkpoint loading/hashing and small deterministic parity batches, so multi-GPU sharding is kept out of the validated path.

Requirements: Kaggle Internet **ON**, a GPU accelerator, and Kaggle Secrets named `HF_TOKEN` and `GITHUB_TOKEN`. Publication is fail-closed: Hugging Face upload only runs after every release gate passes.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import json
import os
import subprocess
import sys

BRANCH = "codex/granite-clm-preview-3b-release"
REPO = Path("/kaggle/working/mini-cells")
WORK = Path("/kaggle/working/granite-clm-preview-3b-work")
OUT = REPO / "artifacts/releases/granite-clm-preview-3b"
HF_REPO = "archelabs-org/native-clm-v0"
HF_SUBDIR = "granite-clm-preview-3b"
REQUIRED_TRANSFORMERS = "4.47.0"
os.environ.setdefault("HF_HOME", "/kaggle/working/hf-cache")

def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), check=True, **kwargs)

if not (REPO / ".git").exists():
    run(["git", "clone", "--branch", BRANCH, "https://github.com/ArcheLabs/mini-cells.git", REPO])
os.chdir(REPO)
run(["git", "fetch", "origin"])
run(["git", "checkout", BRANCH])
run(["git", "pull", "--ff-only", "origin", BRANCH])

# Install the repository without the loose LM extra first, then install the exact
# Transformers generation recorded by the Granite 3.1 3B-A800M checkpoint.
run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"])
run([
    sys.executable, "-m", "pip", "install",
    f"transformers=={REQUIRED_TRANSFORMERS}",
    "huggingface_hub>=0.23.2,<1.0",
    "accelerate>=0.30,<2.0",
])
run([sys.executable, "-m", "pip", "check"])

import torch
import transformers

assert transformers.__version__ == REQUIRED_TRANSFORMERS, (
    f"Expected transformers {REQUIRED_TRANSFORMERS}, got {transformers.__version__}. "
    "Restart the Kaggle session/kernel and Run All."
)
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print(json.dumps({
    "branch": subprocess.check_output(["git", "branch", "--show-current"], text=True).strip(),
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "huggingface_hub": version("huggingface_hub"),
    "accelerate": version("accelerate"),
    "cuda": torch.cuda.is_available(),
    "visible_gpu_count": torch.cuda.device_count(),
    "visible_gpus": gpu_names,
    "execution_policy": "single_gpu_parity",
    "selected_device": "cuda:0",
}, indent=2))
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."


In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["GITHUB_TOKEN"] = secrets.get_secret("GITHUB_TOKEN")
assert os.environ["HF_TOKEN"], "Missing Kaggle Secret: HF_TOKEN"
assert os.environ["GITHUB_TOKEN"], "Missing Kaggle Secret: GITHUB_TOKEN"
print("Secrets loaded (values not printed).")


In [ ]:
# Resolve and freeze the exact upstream revision used by this release, then run a
# cheap config-only compatibility preflight before downloading/materializing weights.
from huggingface_hub import HfApi
from transformers import AutoConfig

MODEL_ID = "ibm-granite/granite-3.1-3b-a800m-base"
info = HfApi().model_info(MODEL_ID)
PINNED_REVISION = info.sha
assert PINNED_REVISION and len(PINNED_REVISION) >= 7
cfg = AutoConfig.from_pretrained(MODEL_ID, revision=PINNED_REVISION)
assert cfg.model_type == "granitemoe", cfg.model_type
assert "GraniteMoeForCausalLM" in list(getattr(cfg, "architectures", []) or [])
assert str(getattr(cfg, "transformers_version", "")) == REQUIRED_TRANSFORMERS, (
    getattr(cfg, "transformers_version", None), REQUIRED_TRANSFORMERS
)
print(json.dumps({
    "source_model": MODEL_ID,
    "pinned_revision": PINNED_REVISION,
    "checkpoint_transformers_version": getattr(cfg, "transformers_version", None),
    "model_type": cfg.model_type,
    "architectures": cfg.architectures,
    "layers": cfg.num_hidden_layers,
    "experts": cfg.num_local_experts,
    "top_k": cfg.num_experts_per_tok,
    "hf_target": f"{HF_REPO}/{HF_SUBDIR}",
}, indent=2))


In [ ]:
# Lossless lift -> verification -> metric/provenance recording -> Hugging Face publication.
# The script refuses to upload if the pinned loader environment or any release gate fails.
run([
    sys.executable,
    "scripts/research/run_granite_clm_preview_3b_release.py",
    "--model-id", MODEL_ID,
    "--revision", PINNED_REVISION,
    "--hf-repo", HF_REPO,
    "--hf-subdir", HF_SUBDIR,
    "--work-dir", WORK,
    "--output-dir", OUT,
    "--device", "cuda:0",
    "--dtype", "float16",
    "--tolerance", "1e-5",
    "--copy-mode", "hardlink",
    "--publish-hf",
])


In [ ]:
# Render the durable release evidence.
metrics = json.loads((OUT / "metrics.json").read_text())
provenance = json.loads((OUT / "provenance.json").read_text())
parity = json.loads((OUT / "parity_report.json").read_text())
hf_publish = json.loads((OUT / "hf_publish.json").read_text())

summary = {
    "status": metrics["status"],
    "source_revision": provenance["source"]["revision"],
    "hf_target": provenance["hf_target"],
    "hf_commit": hf_publish.get("commit_oid"),
    "manifest_identity_sha256": provenance["manifest_identity_sha256"],
    "layers": metrics["model"]["num_hidden_layers"],
    "experts": metrics["model"]["num_local_experts"],
    "top_k": metrics["model"]["num_experts_per_tok"],
    "expert_address_count": metrics["conversion"]["expert_address_count"],
    "max_abs_logit_error": parity.get("max_abs_logit_error"),
    "max_abs_router_error": parity.get("max_abs_router_error"),
    "router_topk_identity": parity.get("gates", {}).get("router_topk_identity"),
    "greedy_token_identity": parity.get("gates", {}).get("greedy_token_identity"),
    "transformers": metrics["environment"]["transformers_version"],
    "visible_gpu_count": metrics["environment"]["cuda_device_count"],
    "execution_policy": metrics["environment"]["execution_policy"],
    "runner_seconds": metrics["runtime"]["runner_seconds"],
}
print(json.dumps(summary, indent=2))
assert metrics["status"] == "PASS"
assert metrics["environment"]["transformers_version"] == REQUIRED_TRANSFORMERS
assert provenance["source"]["revision"] == PINNED_REVISION
assert hf_publish.get("published") is True
print("\n--- RESULTS.md ---\n")
print((OUT / "RESULTS.md").read_text())


In [ ]:
# Push only lightweight release evidence to the dedicated GitHub branch.
run([
    sys.executable,
    "scripts/research/publish_granite_clm_preview_3b.py",
    "--branch", BRANCH,
])
print("Granite-CLM-Preview-3B: Hugging Face bundle and GitHub evidence published.")
